In [95]:
using LowLevelFEM

In [96]:
line_mesh()

mat = Material("body")
Pu = Problem([mat], type=:ScalarField, dim=2, field=:u, rhs_field=:N)
Pv = Problem([mat], type=:ScalarField, dim=2, field=:v, rhs_field=:T)
Pφ = Problem([mat], type=:ScalarField, dim=2, field=:φ, rhs_field=:M)

Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 11, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :M)

In [97]:
E = mat.E
G = mat.μ
I = 1
κ = 2 / 3
A = 1

1

In [98]:
Kuu = ∫(AxialGrad(Pu) ⋅ (E * A) ⋅ AxialGrad(Pu); Γ="body")

Kvv = ∫(AxialGrad(Pv) ⋅ (κ * G * A) ⋅ AxialGrad(Pv); Γ="body")

Kvφ = ∫(AxialGrad(Pv) ⋅ (-κ * G * A) ⋅ Id(Pφ); Γ="body")

Kφv = ∫(Id(Pφ) ⋅ (-κ * G * A) ⋅ AxialGrad(Pv); Γ="body")

Kφφ = ∫(Id(Pφ) ⋅ (κ * G * A) ⋅ Id(Pφ); Γ="body") +
      ∫(AxialGrad(Pφ) ⋅ (E * I) ⋅ AxialGrad(Pφ); Γ="body")

sparse([1, 3, 2, 11, 1, 3, 4, 3, 4, 5  …  9, 8, 9, 10, 9, 10, 11, 2, 10, 11], [1, 1, 2, 2, 3, 3, 3, 4, 4, 4  …  8, 9, 9, 9, 10, 10, 10, 11, 11, 11], [2.0017094017094017e6, -1.9991452991452992e6, 2.001709401709403e6, -1.9991452991453004e6, -1.9991452991452992e6, 4.0034188034188035e6, -1.9991452991452992e6, -1.9991452991452992e6, 4.0034188034188026e6, -1.9991452991452985e6  …  -1.9991452991453004e6, -1.9991452991453004e6, 4.003418803418806e6, -1.9991452991453004e6, -1.9991452991453004e6, 4.003418803418806e6, -1.9991452991453004e6, -1.9991452991453004e6, -1.9991452991453004e6, 4.003418803418806e6], 11, 11)

In [99]:
Zuv = 0 * ∫(Id(Pu) ⋅ Id(Pv); Γ="body")
Zuφ = 0 * ∫(Id(Pu) ⋅ Id(Pφ); Γ="body")

sparse(Int64[], Int64[], Float64[], 11, 11)

In [100]:
K = SystemMatrix([
    Kuu Zuv Zuφ
    Zuv' Kvv Kvφ
    Zuφ' Kφv Kφφ
])
K[:, :]

33×33 SparseArrays.SparseMatrixCSC{Float64, Int64} with 145 stored entries:
⎡⠕⣥⡀⠀⠀⠂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠈⠻⣦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠠⠀⠀⠈⠻⢆⢀⠀⠀⠀⠀⡀⡀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠐⠱⣦⡀⠀⠈⠊⢴⣄⠀⠀⠁⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠈⠻⣦⡀⠀⠀⠑⢷⢄⠀⎥
⎢⠀⠀⠀⠀⠀⠠⡢⠀⠀⠈⢛⢔⠄⠀⠀⠑⡁⎥
⎢⠀⠀⠀⠀⠀⠈⠐⢷⢄⠀⠀⠁⠻⣦⡀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠙⢗⢄⠀⠀⠈⠻⣦⡀⎥
⎣⠀⠀⠀⠀⠀⠀⠁⠀⠀⠀⠁⠈⠀⠀⠀⠈⠁⎦

In [101]:
supp_u = BoundaryCondition("left", problem=Pu, u=0)
supp_v = BoundaryCondition("left", problem=Pv, v=0)
supp_φ = BoundaryCondition("left", problem=Pφ, φ=0)

BoundaryCondition("left", Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 11, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :M), Dict{Symbol, Union{Function, Number, ScalarField}}(:φ => 0))

In [102]:
fu = ∫(Pu ⋅ 0, Γ="right")
fv = ∫(Pv ⋅ 1, Γ="right")
fφ = ∫(Pφ ⋅ 0, Γ="right")

nodal ScalarField
[0.0; 0.0; … ; 0.0; 0.0;;]

In [103]:
F = SystemVector([fu, fv, fφ])

SystemVector([0.0; 0.0; … ; 0.0; 0.0;;], nothing, Problem[Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 11, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :N), Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 11, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :T), Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 11, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :M)], [0, 11, 22])

In [104]:
u, v, φ = solveField(K, F, support=[supp_u, supp_v, supp_φ])

(ScalarField(Matrix{Float64}[], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0], Int64[], 1, :scalar, Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 11, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :N)), ScalarField(Matrix{Float64}[], [0.0; 2.1162144840845943e-5; … ; 1.6769750053407362e-5; 1.896344798120057e-5;;], [0.0], Int64[], 1, :scalar, Problem("line_mesh", :ScalarField, 2, 1, Material[Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 11, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :v, :T)), ScalarField(Matrix{Float64}[], [0.0; 2.4994659260841674e-6; … ; 2.399487289040801e-6; 2.474471266823326e-6;;], [0.0], Int64[], 1, :scalar, Problem("line_mesh", :ScalarField, 2, 1, Material[Mat

In [105]:
u0 = showDoFResults(u, name="u")
v0 = showDoFResults(v, name="v")
φ0 = showDoFResults(φ, name="φ")

2

In [106]:
v1 = ScalarField([], v.a, u.t, [], u.nsteps, u.type, u.model)
disp = VectorField([u, v1, 0v1])

showDoFResults(disp)
plotOnPath("body", v0)
plotOnPath("body", φ0)

5

In [ ]:
openPostProcessor()